# BRCA1 — structured graph vs. literature (kg-rag R8)

One gene, two channels, **one shared identifier**:

- **Channel A — structured** ([bioinsight-graph](https://github.com/LordKay-sudo/bioinsight-graph)): ranked disease associations + evidence scores for `BRCA1`.
- **Channel B — literature** (this repo, kg-rag): cited quotes retrieved from the document corpus.

Both resolve `BRCA1` to the same Ensembl id **`ENSG00000012048`**, so the structured row and the literature quote describe the same entity.

> Demo data: synthetic corpus + frozen slice. Associations are illustrative, not causal.

In [ ]:
from __future__ import annotations

import os

import httpx
import pandas as pd
from IPython.display import display

KG_RAG_API = os.getenv("KG_RAG_API_URL", "http://localhost:8001/api/v1").rstrip("/")
BIOINSIGHT_API = os.getenv("BIOINSIGHT_API_URL", "http://localhost:8000/api/v1").rstrip("/")
GENE_SYMBOL = os.getenv("GENE_SYMBOL", "BRCA1")
ENSG_ID = "ENSG00000012048"  # shared join key for BRCA1

print(f"kg-rag:     {KG_RAG_API}")
print(f"bioinsight: {BIOINSIGHT_API}")
print(f"gene:       {GENE_SYMBOL} ({ENSG_ID})")

## Channel A — structured associations (BioInsight)

Optional: if BioInsight is offline, this cell reports it and the notebook continues with literature only.

In [ ]:
df_struct = pd.DataFrame()
try:
    with httpx.Client(base_url=BIOINSIGHT_API, timeout=15.0) as client:
        diseases = client.get(f"/genes/{ENSG_ID}/diseases").raise_for_status().json()
    rows = [
        {"disease": d.get("disease_name"), "disease_id": d.get("disease_id"), "score": d.get("score")}
        for d in diseases.get("diseases", [])
    ]
    df_struct = pd.DataFrame(rows).sort_values("score", ascending=False).head(10)
    display(df_struct)
except Exception as exc:
    print(f"BioInsight unavailable ({exc}). Continuing with literature channel only.")

## Channel B — cited literature (kg-rag)

Bias the ask toward the shared id so retrieval favors graph-aligned chunks (R6).

In [ ]:
with httpx.Client(base_url=KG_RAG_API, timeout=60.0) as client:
    ask = client.post(
        "/ask",
        json={
            "question": f"What is the link between {GENE_SYMBOL} and disease?",
            "gene_id": ENSG_ID,
        },
    ).raise_for_status().json()

print("Answer:\n", ask["answer"], "\n")

cite_rows = [
    {
        "document": c.get("document_title") or c.get("document_id"),
        "quote": (c.get("snippet") or "")[:160],
        "reference": c.get("reference_url"),
    }
    for c in ask.get("citations", [])
]
df_lit = pd.DataFrame(cite_rows)
display(df_lit)

## Entities resolve to the shared identifier

The `ontology_id` returned by kg-rag is the same key BioInsight uses — that is what lets the two channels line up.

In [ ]:
ent_rows = [
    {"type": e["type"], "id": e["id"], "ontology_id": e.get("ontology_id")}
    for e in ask.get("entities", [])
]
df_ent = pd.DataFrame(ent_rows)
display(df_ent)

brca1 = next((e for e in ask.get("entities", []) if e["id"].upper() == GENE_SYMBOL.upper()), None)
if brca1:
    print(f"kg-rag resolved {GENE_SYMBOL} -> {brca1.get('ontology_id')}")
    print(f"BioInsight join key             -> {ENSG_ID}")
    print("match:", brca1.get("ontology_id") == ENSG_ID)

## Side by side

Structured score (if available) next to the literature evidence for the same gene.

In [ ]:
print(f"=== {GENE_SYMBOL} ({ENSG_ID}) ===\n")

if not df_struct.empty:
    print("Structured (BioInsight) — top disease associations:")
    for _, r in df_struct.head(5).iterrows():
        score = f"{r['score']:.3f}" if pd.notna(r["score"]) else "n/a"
        print(f"  {score}  {r['disease']}")
else:
    print("Structured (BioInsight): unavailable")

print("\nLiterature (kg-rag) — cited quotes:")
for _, r in df_lit.iterrows():
    print(f"  [{r['document']}] {r['quote']}")
    if r["reference"]:
        print(f"      ref: {r['reference']}")

## Audit the literature source (R3)

Open every chunk + extraction provenance behind the first cited document.

In [ ]:
if ask.get("citations"):
    doc_id = ask["citations"][0]["document_id"]
    with httpx.Client(base_url=KG_RAG_API, timeout=15.0) as client:
        audit = client.get(f"/documents/{doc_id}/chunks").raise_for_status().json()
    print(f"{audit['document_id']} — {audit.get('title')} ({audit['chunk_count']} chunks)")
    prov = [
        {
            "chunk": c["chunk_id"],
            "entity": e["id"],
            "ontology_id": e.get("ontology_id"),
            "confidence": e.get("confidence"),
            "extractor": e.get("extractor_version"),
        }
        for c in audit["chunks"]
        for e in c["entities"]
    ]
    display(pd.DataFrame(prov))

## Takeaway

- **Structured** gives ranked, scored associations; **literature** gives cited prose.
- The **shared ENSG id** is the join key — same gene in both systems.
- Every literature claim is auditable down to chunk + extractor confidence (R3/R7).
- Next: an agent can fuse both channels (see embabel-mcp `graph-and-literature` / M8).